# 02 - Convolutional VAE + k-means

Learns a 16-dimensional summary of each daily scalogram (128 scales x 24 hours). K-means on those summaries plus the daily price level assigns each day to a market regime.

Pipeline: scalogram -> ConvVAE encoder -> latent mean (mu) + daily price level -> k-means -> regime

Methodology notes:
1. Days are split chronologically (60% train, 20% validation, 20% test). The 2023-2024 El Nino falls in the test period.
2. The k-means is fitted on training days only and then applied to all days.
3. Scalograms come from the real price with each day's mean removed, so they show only the intraday shape. `real_price_log` uses the log real price and `real_price` the real price in COP/kWh. The daily level (mean and standard deviation of the same series) is added to the latent vector before clustering.
4. The VAE is trained once. K-means then runs once for each k in `K_VALUES`, and each k gets its own run folder in `experiments/conv_vae/<clustering>/` and its own row in `experiments/registry.csv`. All folders of one execution share the same model, history and latents, and `trained_in` in their config names the first one.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import torch

import numpy as np
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

plt.rcParams['font.family'] = 'Arial'
alt.data_transformers.enable('vegafusion')

from src.data import load_scalograms, load_context, chronological_split, ScalogramDataset
from src.models.conv_vae import ConvVAE
from src.training import set_seed, train
from src.clustering import with_level, level_columns, select_k, fit_clusters, order_regimes, summary_metrics, regime_profile, save_regimes
from src.plots import loss_curves, selection_chart, tsne_chart, regime_timeline, plot_reconstructions
from src import experiments as exp

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Experiment configuration

In [ ]:
SCALOGRAM_SET = "real_price_log"
CLUSTERING = "kmeans"
K_VALUES = [2, 3, 4, 5, 6]

CONFIG = {
    "model": "conv_vae",
    "scalogram_set": SCALOGRAM_SET,
    "clustering": CLUSTERING,
    "k_values": K_VALUES,
    "k_range": [2, 3, 4, 5, 6, 7, 8],
    "z_dim": 16,
    "filters": [128, 64, 32],
    "beta": 1.0,
    "kl_warmup_epochs": 0,
    "optimizer": "Adam",
    "lr": 1e-3,
    "batch_size": 64,
    "epochs": 30,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "seed": 42,
}

## 1. Run folders

One folder per k, created before training.

In [ ]:
SCALOGRAM_DIR = Path("../data/processed/scalograms") / SCALOGRAM_SET
DATA_DIR = Path("../data/processed")

RUN_DIRS = {k: exp.create_run(CLUSTERING, SCALOGRAM_SET, CLUSTERING, k, root=exp.EXPERIMENTS_DIR / CONFIG["model"])
            for k in K_VALUES}
TRAINED_IN = exp.run_id(RUN_DIRS[K_VALUES[0]])
RUN_CONFIGS = {k: {**CONFIG, "n_regimes": k, "trained_in": TRAINED_IN} for k in K_VALUES}
for k, run_dir in RUN_DIRS.items():
    exp.save_config(run_dir, RUN_CONFIGS[k])


def save_all(save, *args, **kwargs):
    """Call save(run_dir, ...) for every run folder."""
    for run_dir in RUN_DIRS.values():
        save(run_dir, *args, **kwargs)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(CONFIG["seed"])
for k, run_dir in RUN_DIRS.items():
    print(f"k={k}: {exp.run_id(run_dir)}")
print(f"Device: {DEVICE}")

## 2. Data

In [ ]:
images, metadata = load_scalograms(SCALOGRAM_DIR)
context = load_context(DATA_DIR, metadata)
train_idx, val_idx, test_idx = chronological_split(len(images), CONFIG["train_ratio"], CONFIG["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

IMG_C, IMG_H, IMG_W = images.shape[1:]
print(f"Scalograms ({SCALOGRAM_SET}): {images.shape}, range [{images.min():.3f}, {images.max():.3f}]")
print(f"Daily level columns: {level_columns(metadata)}")
split_table = exp.split_table(context, {"train": train_idx, "val": val_idx, "test": test_idx})
print(split_table)
save_all(exp.save_table, "splits", split_table, show=False)

In [ ]:
train_loader = DataLoader(ScalogramDataset(images, train_idx), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(ScalogramDataset(images, val_idx), batch_size=CONFIG["batch_size"], shuffle=False)

In [ ]:
def bce_floor(x):
    x = np.clip(x.reshape(len(x), -1).astype(np.float64), 1e-7, 1 - 1e-7)
    return -(x * np.log(x) + (1 - x) * np.log(1 - x)).sum(axis=1)

for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    f = bce_floor(images[idx])
    print(f"{name}: floor {f.mean():.0f}, mean pixel {images[idx].mean():.3f}")

## 3. Model

In [ ]:
model = ConvVAE(CONFIG["z_dim"], IMG_C, IMG_H, IMG_W, CONFIG["filters"]).to(DEVICE)

with torch.no_grad():
    out_shape = tuple(model(torch.from_numpy(images[:2]).to(DEVICE))[0].shape[1:])
assert out_shape == (IMG_C, IMG_H, IMG_W), f"decoder output {out_shape} does not match input"

N_PARAMS = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Encoder feature map: {model.encoder.feat_shape}")
print(f"Decoder output: {out_shape}")
hyperparameters = exp.hyperparameter_table(CONFIG, N_PARAMS)
print(hyperparameters)
save_all(exp.save_table, "hyperparameters", hyperparameters, show=False)

## 4. Training

In [ ]:
history = train(model, train_loader, val_loader, DEVICE, epochs=CONFIG["epochs"], lr=CONFIG["lr"],
                beta=CONFIG["beta"], kl_warmup_epochs=CONFIG["kl_warmup_epochs"])
save_all(exp.save_history, history)
save_all(exp.save_model, model)

## 5. Loss curves

In [ ]:
loss_plot = loss_curves(history, components=("total", "recon", "kl"))
loss_plot

In [ ]:
for run_dir in RUN_DIRS.values():
    loss_plot.save(exp.figure_path(run_dir, "loss_curves"), scale_factor=2.0)

## 6. Reconstructions

Test-period days.

In [ ]:
model.eval()
show_idx = test_idx[np.linspace(0, len(test_idx) - 1, 8).astype(int)]
with torch.no_grad():
    recon = model(torch.from_numpy(images[show_idx]).to(DEVICE))[0].cpu().numpy()

fig = plot_reconstructions(images[show_idx], recon, [str(d.date()) for d in context["date"].iloc[show_idx]])
for run_dir in RUN_DIRS.values():
    fig.savefig(exp.figure_path(run_dir, "reconstructions"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Latent vectors

In [ ]:
latents = model.extract_latents(images, DEVICE)
save_all(exp.save_latents, latents)
features = with_level(latents, metadata, train_idx)

active = int((latents.var(axis=0) > 0.01).sum())
print(f"Latent vectors: {latents.shape}")
print(f"Final train KL: {history[-1]['train_kl']:.2f}")
print(f"Active dimensions: {active} of {CONFIG['z_dim']}")
print(f"Clustering features (latents + daily level): {features.shape}")
print("Latent variance per dim:", np.round(latents.var(axis=0), 5))

## 8. Regimes with k-means

K-means uses the latent vector plus the daily price level (mean and standard deviation, z-scored on training days), the same input as the GMM notebooks, because the scalograms only carry the intraday shape. It is fitted on the training days only and then assigns every day. Regimes are numbered by ascending mean real price, so regime 0 is always the cheapest and labels are comparable across models.

The selection table is the same for every k, so it is computed once. It shows, for each k on the training days, the inertia (look for the elbow), silhouette (higher is better), Davies-Bouldin (lower is better) and Calinski-Harabasz (higher is better).

In [ ]:
selection = select_k(features, train_idx, CLUSTERING, CONFIG["k_range"], CONFIG["seed"])
selection_table = exp.selection_table(selection)
print(selection_table)
save_all(exp.save_table, "k_selection", selection_table, show=False)
selection_chart(selection, K_VALUES[0])

## 9. One run per k

The t-SNE map depends only on the features, so it is computed once and colored with each k's regimes. For each k the loop saves to its run folder:
- figures: k selection, t-SNE, regime timeline (plus the loss curves and reconstructions saved above)
- tables: regime profile, clustering metrics, price spread by regime, share of each weekday in each regime, and the feature checks (level only, latents only, latents + level, 7-day mean latents + level)
- `regimes.csv`, `metrics.json` and its row in `registry.csv`

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])
plot_idx = np.sort(rng.choice(len(features), size=min(5000, len(features)), replace=False))
coords = TSNE(n_components=2, perplexity=30, random_state=CONFIG["seed"]).fit_transform(features[plot_idx])

In [ ]:
price = context["precio_real_cop_kwh"].to_numpy()
level = with_level(np.zeros((len(latents), 0)), metadata, train_idx)
smooth = with_level(pd.DataFrame(latents).rolling(7, min_periods=1).mean().to_numpy(), metadata, train_idx)
FEATURE_SETS = (("level only", level), ("latents only", latents), ("latents + level", features),
                ("7d-mean latents + level", smooth))
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


def feature_checks(k):
    t = PrettyTable(["Features", "Silhouette", "Davies-Bouldin", "Mean run", "Switches / yr", "Median price by regime"])
    for name, f in FEATURE_SETS:
        lab = order_regimes(fit_clusters(f, train_idx, k, CLUSTERING, CONFIG["seed"])[0], price)
        m = summary_metrics(f, lab, seed=CONFIG["seed"])
        med = context.assign(r=lab).groupby("r")["precio_real_cop_kwh"].median().round(0).tolist()
        t.add_row([name, f"{m['silhouette']:.3f}", f"{m['davies_bouldin']:.3f}", f"{m['mean_run_days']:.1f}",
                   f"{m['switches_per_year']:.0f}", med])
    return t


RESULTS = {}
for k, run_dir in RUN_DIRS.items():
    labels, probs, _ = fit_clusters(features, train_idx, k, CLUSTERING, CONFIG["seed"])
    labels = order_regimes(labels, price)
    ctx = context.assign(regime=labels)

    selection_chart(selection, k).save(exp.figure_path(run_dir, "k_selection"), scale_factor=2.0)
    tsne_chart(coords, labels[plot_idx], f"t-SNE of the latent space (k={k})").save(
        exp.figure_path(run_dir, "tsne_regimes"), scale_factor=2.0)
    regime_timeline(ctx, split_dates, title=f"Detected regimes over time (k={k})").save(
        exp.figure_path(run_dir, "regime_timeline"), scale_factor=2.0)

    profile = regime_profile(ctx)
    metrics = {"run_id": exp.run_id(run_dir), "model": CONFIG["model"], "scalogram_set": SCALOGRAM_SET,
               "clustering": CLUSTERING, "k": k, **summary_metrics(features, labels, seed=CONFIG["seed"])}
    save_regimes(run_dir, context, labels, None, metrics)
    exp.log_run(run_dir, RUN_CONFIGS[k], metrics, registry=exp.EXPERIMENTS_DIR / "registry.csv")

    spread = ctx.groupby("regime")["precio_real_cop_kwh"].describe(percentiles=[0.1, 0.5, 0.9])
    weekday = pd.crosstab(labels, ctx["date"].dt.dayofweek, normalize="columns").mul(100).round(0)
    weekday.index.name, weekday.columns = "regime", WEEKDAYS
    exp.save_table(run_dir, "regime_profile", exp.regime_profile_table(profile), show=False)
    exp.save_table(run_dir, "clustering_metrics", exp.metrics_table(metrics), show=False)
    exp.save_table(run_dir, "price_spread", spread[["count", "10%", "50%", "90%", "min", "max"]].round(1), show=False)
    exp.save_table(run_dir, "weekday_share", weekday, show=False)
    exp.save_table(run_dir, "feature_checks", feature_checks(k), show=False)

    RESULTS[k] = {"metrics": metrics, "profile": profile, "weekday": weekday, "context": ctx}
    print(f"k={k}: {exp.run_id(run_dir)}  silhouette={metrics['silhouette']:.3f}  "
          f"mean run={metrics['mean_run_days']:.1f} d  switches/yr={metrics['switches_per_year']:.1f}")

## 10. Comparison across k

One row per k. "Top regime" is the most expensive regime (highest number). The last two columns are the share of El Nino days (ONI >= 0.5) in 2015-2016 and in 2023-2024 that fall in the top regime. Read them against `top_share_pct`: if the top regime holds 76% of all days, holding 76% of El Nino days says nothing. A regime that tracks scarcity holds a much larger share of El Nino days than of all days, and switches few times per year.

In [ ]:
def nino_share_in_top(ctx, years):
    days = ctx[(ctx["enso_phase"] == "El Nino") & ctx["date"].dt.year.isin(years)]
    return 100 * (days["regime"] == ctx["regime"].max()).mean()


rows = []
for k, r in RESULTS.items():
    m, top = r["metrics"], r["profile"].iloc[-1]
    rows.append({
        "k": k, "silhouette": m["silhouette"], "davies_bouldin": m["davies_bouldin"],
        "calinski_harabasz": m["calinski_harabasz"], "mean_run_days": m["mean_run_days"],
        "switches_per_year": m["switches_per_year"], "top_share_pct": top["share_pct"],
        "top_price": top["mean_real_price"], "top_reservoir": top["mean_reservoir"],
        "nino_2015_16_in_top_pct": nino_share_in_top(r["context"], [2015, 2016]),
        "nino_2023_24_in_top_pct": nino_share_in_top(r["context"], [2023, 2024]),
    })
k_sweep = pd.DataFrame(rows).set_index("k").round(3)
save_all(exp.save_table, "k_sweep", k_sweep, show=False)
k_sweep

In [ ]:
for k, r in RESULTS.items():
    print(f"k={k}  {exp.run_id(RUN_DIRS[k])}")
    print(exp.regime_profile_table(r["profile"]))
    print("Share of each weekday in each regime (%)")
    print(r["weekday"].to_string(), end="\n\n")

Regime strip for every k, one row per k.

In [ ]:
alt.vconcat(*[
    alt.Chart(r["context"].assign(regime=r["context"]["regime"].astype(str))).mark_tick(thickness=1, size=12).encode(
        x=alt.X("date:T", title=""), y=alt.Y("regime:O", title=f"k={k}"), color=alt.Color("regime:N", legend=None),
    ).properties(width=800, height=18 * k)
    for k, r in RESULTS.items()
]).resolve_scale(color="independent")